---
# Chapter 2 — The Measurement Instrument

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 2: The Measurement Instrument |
| Central question | How do we know whether an AI memory is any good? |
| Main concepts | Memory Measurement Instrument, Task-metric separation, Controlled corpus, Real corpus, Baseline ladder, Failure taxonomy, Instrument validity |
| Implementation | experiments.benchmark.memory_measurement |
| Experiment | memory_measurement v0.1 scorers (deterministic) |
| Evidence status | Instrument defined; experiment pending |
| Depends on | Chapter 1 (behavioural definition) |

---

## What this notebook demonstrates

This chapter defines the **Memory Measurement Instrument** — the fixed apparatus used to determine whether a memory system remembers. The notebook:

1. **Loads the actual instrument implementation** from `experiments/benchmark/memory_measurement`
2. **Constructs a `MemoryTask`, hidden ground truth, `SystemOutput`, and real scorer output**
3. **Shows why task, truth, evidence, behaviour and score are separate objects**
4. **Demonstrates the deterministic scorers** (source recall, decision exactness, temporal accuracy, supersession, abstention, unsupported sources)
5. **Shows the frozen March-to-October demonstration** over 6 tasks against 3 canned systems

> **Evidence status**: The v0.1 scorers are implemented and runnable. The controlled-world generator, frozen fixtures, full query sets, and metric-behaviour bridge remain pending (see Chapter 2 limitations).

## The chapter question

> **How do we know whether an AI memory is any good?**

The instrument answers four questions:

1. **What do we measure?** — Preservation, retrieval, reconstruction, provenance, temporal correctness, epistemic correctness, groundedness, abstention, context selection, behavioural influence, harm, cost
2. **How do we measure them?** — Fixed tasks, fixtures, controls, metrics that don't reward fluency
3. **How do we score them?** — Success, partial success, failure, abstention, hallucination, regression, harm at claim-edge granularity
4. **How do we know the measurement is valid?** — Oracle controls, held-out splits, fluent-summariser rejection, metric-behaviour bridge

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(2)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 2 Concepts")

## The running example

The chapter uses a **March-to-October event-store demonstration** contrasting three canned systems:

- **Good system**: Uses current decision (PostgreSQL) with evidence
- **Stale system**: Reports superseded decision (SQLite) as current
- **Fluent-but-wrong system**: Names PostgreSQL but fabricates rationale

These run over six task families: locate, decide, justify, historical truth, current truth, abstention.

## The mechanism: Task-Metric Separation

The instrument's core design: **task, hidden ground truth, evidence, expected behaviour, measurements, and scores are distinct objects**. The system receives prompt + history reference; expected fields remain evaluator-only.

In [ ]:
# Load the actual instrument implementation
from experiments.benchmark.memory_measurement import (
    MemoryTask,
    SystemOutput,
    HistoryItem,
    score_task,
    FailureClass,
    MemoryObservation,
)
from experiments.benchmark.memory_measurement.tasks import (
    ANSWERABLE,
    UNANSWERABLE,
)

print("Instrument classes loaded successfully")
print(f"MemoryTask fields: {MemoryTask.__dataclass_fields__.keys()}")
print(f"SystemOutput fields: {SystemOutput.__dataclass_fields__.keys()}")

## Construct a MemoryTask with hidden ground truth

The key separation: the task carries the prompt and history reference, while **expected_* fields are evaluator ground truth the system must never receive**.

In [ ]:
# Create a task for the event-store decision
from experiments.benchmark.memory_measurement import MemoryTask

task = MemoryTask(
    task_id="decision-event-store",
    family="decision",
    prompt="What should new event-store services use?",
    history_ref="controlled-v0.1",
    expected_sources=("adr-007",),
    expected_state="PostgreSQL",
    expected_aliases=("PostgreSQL 14", "PG"),
    superseded_options=("SQLite",),
    expected_current="PostgreSQL",
    expected_historical="SQLite",
    temporal_mode="current",
    expected_status=ANSWERABLE,
)

print("Task created:")
print(f"  task_id: {task.task_id}")
print(f"  family: {task.family}")
print(f"  prompt: {task.prompt}")
print(f"  expected_sources: {task.expected_sources}")
print(f"  expected_state: {task.expected_state}")
print(f"  superseded_options: {task.superseded_options}")
print(f"  temporal_mode: {task.temporal_mode}")
print(f"  expected_status: {task.expected_status}")

## Create SystemOutput from a system under test

The system output contains what the system *actually* produced: answer, retrieved sources, cited sources, abstention flag.

In [ ]:
from experiments.benchmark.memory_measurement import SystemOutput

# Simulate a GOOD system output
output_good = SystemOutput(
    task_id="decision-event-store",
    answer="New services should use PostgreSQL per adr-007 (July 11 decision). The team benchmarked SQLite and found it slowed under concurrent writes.",
    retrieved_ids=("adr-007", "session-035", "session-033"),
    cited_sources=("adr-007",),
    abstained=False,
)

# Simulate a STALE system output (reports superseded decision)
output_stale = SystemOutput(
    task_id="decision-event-store",
    answer="New services should use SQLite. The team prototyped it in session-031 and it worked.",
    retrieved_ids=("session-031", "session-033"),
    cited_sources=("session-031",),
    abstained=False,
)

# Simulate a FLUENT-BUT-WRONG system (names right answer, fabricates rationale)
output_fluent_wrong = SystemOutput(
    task_id="decision-event-store",
    answer="PostgreSQL is the right choice because of the Redis benchmark in adr-009.",
    retrieved_ids=("adr-009",),
    cited_sources=("adr-009",),  # adr-009 is the REDIS REJECTION, not PostgreSQL decision
    abstained=False,
)

# Simulate an ABSTAINING system on an answerable task
output_abstain = SystemOutput(
    task_id="decision-event-store",
    answer="",
    retrieved_ids=(),
    cited_sources=(),
    abstained=True,
)

print("Four system outputs created for comparison")

## Run the deterministic scorers

The instrument's mechanical scorers use only string comparison and set arithmetic over ledger identifiers. No model judgements.

In [ ]:
from experiments.benchmark.memory_measurement import score_task

# Available history source IDs (for unsupported source check)
history_ids = ("adr-007", "adr-009", "session-031", "session-033", "session-035", "session-072",
                 "fact-301", "fact-302")

outputs = {
    "GOOD": output_good,
    "STALE": output_stale,
    "FLUENT_WRONG": output_fluent_wrong,
    "ABSTAIN": output_abstain,
}

for name, output in outputs.items():
    print(f"\n{'='*60}")
    print(f"SCORING: {name} system")
    print(f"{'='*60}")
    observations = score_task(task, output, history_ids)
    for obs in observations:
        status = "✓" if obs.value else "✗"
        if isinstance(obs.value, float):
            status = f"{obs.value:.2f}"
        failure = f" [{obs.failure_class}]" if obs.failure_class else ""
        print(f"  {obs.metric:30s} {status}{failure}")
        if obs.evidence:
            for e in obs.evidence:
                print(f"    → {e}")

## What happened?

The scorers expose **different failure modes** that a single aggregate score would hide:

Actual scorer output for the four systems (run the cell above):

| System | Source Recall | Decision | Temporal | Supersession | Unsupported |
|--------|---------------|----------|----------|--------------|-------------|
| GOOD | 1.00 | ✓ | ✓ current | ✗ [MISSED_SUPERSESSION]¹ | 0.00 |
| STALE | 0.00 [CONTEXT_OMISSION] | ✗ [MISSED_SUPERSESSION] | ✗ [STALE_STATE] | ✗ | 0.00 |
| FLUENT_WRONG | 0.00 | ✓ (names PostgreSQL) | ✓ | ✓ | 0.00² |
| ABSTAIN | 0.00 [NOT_RETRIEVED] | ✗ [UNNECESSARY_ABSTENTION] | ✗ | — | — |

- **GOOD** retrieves and decides correctly, but trips the supersession heuristic¹.
- **STALE** retrieves the wrong interval and is diagnosed per-layer: omission, wrong interpretation, stale state.
- **FLUENT_WRONG** names the right answer while citing `adr-009` (the Redis rejection) — yet scores 0.00 unsupported².
- **ABSTAIN** fails because the task *is* answerable.

¹ The v0.1 supersession check is a 60-character negation-window heuristic over answer text (see `scorers.py`, documented limitation): mentioning `SQLite` without a nearby negation token such as *not/instead/rejected* counts as recommending it. Scope errors remain possible; the check is mechanical, not semantic.

² The v0.1 unsupported-source check answers only whether a cited identifier *exists* in history. Whether `adr-009` actually *supports* the PostgreSQL claim is claim-level support — Chapter 7 evidence lineage, not this scorer. The instrument reports what it can check and leaves the rest pending rather than pretending.

This is why **task-metric separation** matters: each failure names the layer to repair, and each scorer states its own boundary.

## Connect this to the experiment

The chapter describes a **March-to-October demonstration** (`ch2-ev9`) running six tasks against three canned systems. Let's check if that frozen run exists.

In [ ]:
from notebooks.memory._support import REPO
import os

# Check for frozen runs from Chapter 2
runs_dir = REPO.experiments / "benchmark" / "runs"
ch2_runs = [d.name for d in runs_dir.iterdir() if d.is_dir() and d.name.startswith("ch2")]
print(f"Chapter 2 frozen runs found: {ch2_runs}")

# Also check for the demo data mentioned in the chapter
# The chapter mentions 'development driver' runs
print("\nNote: Chapter 2 states 'implemented; comparative book experiment pending'")
print("The v0.1 scorers run, but the controlled-world generator and frozen fixtures do not yet exist.")

## What this establishes

- **Demos are not evidence**: Builder-selected cases, single phrasings, fluency judging, no reruns
- **Memory quality is multi-dimensional**: 11+ dimensions, no primary aggregate
- **Task-metric separation prevents circular evaluation**: Hidden ledger, evaluator-only ground truth
- **Controlled + real corpus both required**: Mechanical labels + ecological validation, never merged
- **Failure taxonomy attributes to layer**: Ingestion → retrieval → reconstruction → provenance → temporal → abstention → selection → use → harm → evaluator defect
- **Validity is testable**: Oracle controls, held-out splits, fluent-summariser rejection, metric-behaviour bridge

## What this does NOT establish

- No comparative book result yet (controlled generator, frozen fixtures, versioned query sets pending)
- Model-judged scoring, ranking metrics, support-chain validity, validity intervals, calibration, epistemic promotion, behavioural deltas, harm rates, cost accounting are **specified interfaces with pending scorecard fields**
- Metric-behaviour bridge is a falsification plan, not a plot — no frozen runs validating the instrument

## Try it yourself

Create a new `MemoryTask` for a different family (e.g., `provenance` or `temporal`) and score a system output against it. The instrument supports these families:

- `locate` — Where did we discuss X?
- `decision` — What did we decide about X?
- `provenance` — Why did we decide X?
- `temporal` — Is X still true? (current/historical modes)
- `unfinished` — What did we leave unfinished?
- `selection` — What from the past matters right now?

In [ ]:
# TRY IT YOURSELF: Create a temporal task
from experiments.benchmark.memory_measurement import MemoryTask
from experiments.benchmark.memory_measurement.tasks import ANSWERABLE

# Task: What ran in production on 2024-07-15? (historical mode)
# Expected: SQLite (superseded by PostgreSQL on 2024-07-22)
task_temporal = MemoryTask(
    task_id="production-state-jul15",
    family="temporal",
    prompt="What event store was running in production on July 15, 2024?",
    history_ref="controlled-v0.1",
    expected_sources=("fact-301",),
    expected_historical="SQLite",  # Was true on July 15
    expected_current="PostgreSQL",  # Is true now (after July 22)
    temporal_mode="historical",
    expected_status=ANSWERABLE,
)

# A system that answers correctly for the historical question
output_historical_correct = SystemOutput(
    task_id="production-state-jul15",
    answer="On July 15, 2024, the production event store was SQLite (fact-301). The PostgreSQL cutover happened on July 22.",
    retrieved_ids=("fact-301", "fact-302"),
    cited_sources=("fact-301",),
    abstained=False,
)

# A system that gives the CURRENT state when asked for HISTORICAL
output_historical_stale = SystemOutput(
    task_id="production-state-jul15",
    answer="PostgreSQL is the production event store.",
    retrieved_ids=("fact-302",),
    cited_sources=("fact-302",),
    abstained=False,
)

print("=== HISTORICAL MODE TASK ===")
print(f"Prompt: {task_temporal.prompt}")
print(f"Expected historical: {task_temporal.expected_historical}")
print(f"Expected current: {task_temporal.expected_current}")

for name, output in [("CORRECT_HISTORICAL", output_historical_correct), ("STALE_CURRENT", output_historical_stale)]:
    print(f"\n--- {name} ---")
    obs = score_task(task_temporal, output, history_ids)
    for o in obs:
        val = f"{o.value:.2f}" if isinstance(o.value, float) else o.value
        fail = f" [{o.failure_class}]" if o.failure_class else ""
        print(f"  {o.metric:30s} {val}{fail}")

## Where this leads next

Chapter 3 builds the **simplest possible memory system** (conventional RAG baseline) and subjects it to this instrument. The baseline uses:

- PostgreSQL + pgvector for storage
- Hybrid lexical + dense retrieval with reciprocal rank fusion
- Cross-encoder reranking
- Explicit context assembly with `ContextTrace`
- Capable reader with citations

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)